# 22 — Capstone: An Agent That Calls Everything Through API Management

This capstone combines notebooks 16-21: one conversational turn where
*both* the chat completion and the tool call travel through the same
Azure API Management AI gateway, instead of talking to Azure OpenAI and
an MCP server directly.

Architecturally this mirrors `examples/03_chat_with_tools.py` and
`src/agent_loop.py` from the first part of the course -- only the
transport underneath each call changes.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.config import get_settings

settings = get_settings()
missing = [
    name
    for name, value in [
        ("APIM_GATEWAY_ENDPOINT", settings.apim_gateway_endpoint),
        ("APIM_MCP_SERVER_URL", settings.apim_mcp_server_url),
    ]
    if not value
]
if missing:
    print(f"Set {', '.join(missing)} in your .env before running this notebook for real.")
else:
    print("Gateway chat endpoint and MCP server endpoint are both configured.")

In [ ]:
import json

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

from src.apim_mcp_client import connect_apim_mcp

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
)
gateway_client = AzureOpenAI(
    azure_endpoint=settings.apim_gateway_endpoint,
    api_version="2024-04-01-preview",
    azure_ad_token_provider=token_provider,
)

async def get_gateway_tools_schema():
    async with connect_apim_mcp(
        settings.apim_mcp_server_url,
        subscription_key=settings.apim_subscription_key,
    ) as session:
        tools = (await session.list_tools()).tools
        return [
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.inputSchema,
                },
            }
            for tool in tools
        ]

tools_schema = await get_gateway_tools_schema()
print(f"{len(tools_schema)} tool(s) available through the gateway.")

In [ ]:
async def call_gateway_tool(name: str, arguments: dict):
    async with connect_apim_mcp(
        settings.apim_mcp_server_url,
        subscription_key=settings.apim_subscription_key,
    ) as session:
        return await session.call_tool(name, arguments)

async def run_gateway_turn(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    response = gateway_client.chat.completions.create(
        model=settings.azure_openai_model, messages=messages, tools=tools_schema
    )
    message = response.choices[0].message
    messages.append(message)

    if message.tool_calls:
        for call in message.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = await call_gateway_tool(call.function.name, args)
            content = result.content
            if not isinstance(content, str):
                content = json.dumps(content, default=str)
            messages.append(
                {"tool_call_id": call.id, "role": "tool", "name": call.function.name, "content": content}
            )

    final = gateway_client.chat.completions.create(
        model=settings.azure_openai_model, messages=messages, tools=tools_schema
    )
    return final.choices[0].message.content or ""

answer = await run_gateway_turn("What tools do you have available?")
print(answer)